# 100-Task Failure-Intensive HumanEval Steering Benchmark Suite (`Qwen/Qwen3-4B`)
### Optimized for NVIDIA Tesla T4 | 5-Way Steering Comparison

This benchmark evaluates **`Qwen/Qwen3-4B`** across **100 Curated Failure-Intensive HumanEval Benchmark Tasks** ($100 \times 5 = 500$ total evaluations) with **live evaluation for all 100 tasks** (the pre-loaded Tasks 1-63 metrics from the VibeThinker run belong to a different model and are not reused here).

**Qwen3 notes:** `Qwen/Qwen3-4B` has 36 decoder layers and `d_model=2560`, so Layer 18 is the same mid-depth hook point. Qwen3's hybrid thinking mode is controlled by `ENABLE_THINKING` (passed to `apply_chat_template`). The gating threshold $\tau = 2/\sqrt{d_{\text{model}}}$ is computed from the loaded model config.

---

### 5 Steering Procedures Evaluated:
1. **Vanilla Baseline** ($T=0.0$ greedy decoding without system guardrails or activation hooks)
2. **Negative System Prompting** (Instructional prompt guardrail forbidding syntax, scoping, and logic bugs)
3. **Static Failure Steering** ($\alpha=0.08$, Layer 18 unconditional activation hook repelling $v_{\text{fail}}$)
4. **Adaptive RepGated Steering** ($\alpha=0.08, \tau=0.04$, Layer 18 dynamically gated at theoretical $2\sigma$ noise threshold)
5. **Dual Push-Pull Qwen Trace Steering** ($\alpha=0.08, \beta=0.015$, Layer 18 failure push + reasoning-phase CoT pull during $t \le 200$) ⭐

---

### Performance & Continuous Quality Metrics Captured:
* **Pass@1 Accuracy & 95% Wilson Score Confidence Intervals** ($[p_{\text{low}}, p_{\text{high}}]$)
* **McNemar's Paired Exact Test** ($p$-value & statistical significance $p < 0.05$ vs. Vanilla Baseline)
* **Generation Throughput & Latency** (tokens/sec & compute overhead %)
* **Shannon Information Entropy Diagnostics** ($H(X)$ in bits for cognitive uncertainty)
* **3-Gram Repetition Ratio** ($R_{\text{rep}}$ detecting reasoning loops inside `<think>` tags)
* **Step-by-Step Trajectory Tracking** (per-token entropy and latent cosine margin arrays)
* **AST Failure Category Breakdown** (`SyntaxError`, `NameError`, `TypeError`, `IndexError`, `AssertionError`, `SUCCESS`)
* **Interactive Task Inspector & Trajectory Comparator** (`plot_task_trajectory_comparison`)

## ⚡ Section 1: Tesla T4 & GPU Acceleration Setup

In [ ]:
# Install required dependencies
# Only upgrade transformers (Qwen3 needs >=4.51); leave Colab's preinstalled torch/numpy/pyarrow alone
!pip install -q "transformers>=4.51.0" accelerate datasets

import torch
import os
import sys
import gc

# Enable Tensor Core TF32 acceleration (Ampere+) & cuDNN autotuner for all GPUs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model: {gpu_name}")
    print(f"VRAM Total: {vram_gb:.2f} GB")
    # Check for Turing (Tesla T4) vs Ampere+ architecture
    is_turing = "T4" in gpu_name or "2080" in gpu_name or "TU" in gpu_name
    optimal_dtype = torch.float16 if is_turing or not torch.cuda.is_bf16_supported() else torch.bfloat16
    print(f"Optimal Tensor Precision Selected: {optimal_dtype} ({'Tesla T4 FP16 Tensor Core Hardware' if is_turing else 'Ampere/Ada Native BF16'})")
else:
    optimal_dtype = torch.float32
    print("⚠️ Running on CPU mode. GPU runtime strongly recommended.")

## 📦 Section 2: Core Steering & Continuous Quality Tracking Modules

In [ ]:
import re
import math
import time
import io
import contextlib
import numpy as np
import torch
import torch.nn as nn
from typing import List, Tuple, Dict, Any, Optional, Union
from scipy import stats
from sklearn.svm import SVC

# Qwen3 hybrid reasoning switch: True = <think>...</think> reasoning (mirrors VibeThinker style),
# False = direct answers (much faster, avoids reasoning truncation at TOKEN_BUDGET).
ENABLE_THINKING = True

def fix_seeds(seed: int = 42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

def format_chatml_prompt(tokenizer: Any, prompt: str) -> str:
    if "<|im_start|>" in prompt or "<|user|>" in prompt or "<think>" in prompt:
        return prompt
    system_prompt = (
        "You are an expert Python programmer. Solve the user's coding request directly by writing a clean, "
        "valid, and executable Python function. Keep reasoning brief inside <think>...</think>, and output "
        "the requested Python function inside a ```python ``` block."
    )
    user_content = (
        "Complete the following Python function code. "
        "Provide a complete, executable Python function definition starting with `def ...` inside a ```python ... ``` block.\n\n"
        f"{prompt}"
    )
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None) is not None:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
        except Exception:
            pass
    return f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"

def format_negative_system_prompt(tokenizer: Any, prompt: str) -> str:
    if "<|im_start|>" in prompt or "<|user|>" in prompt or "<think>" in prompt:
        return prompt
    system_prompt = (
        "You are an expert Python programmer. Solve the user's coding request directly by writing a clean, "
        "valid, and executable Python function.\n"
        "STRICT GUARDRAILS:\n"
        "1. Do NOT output unindented code outside the function definition.\n"
        "2. Do NOT leave variables uninitialized or reference undefined names.\n"
        "3. Do NOT produce IndexError, TypeError, or SyntaxError.\n"
        "4. Output ONLY the requested Python function inside a ```python ``` block."
    )
    user_content = f"Complete the following Python function code:\n\n{prompt}"
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None) is not None:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
        except Exception:
            pass
    return f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"

CONTRASTIVE_PAIRS = [
    {
        "buggy": "def add_elements(arr):\n    res = []\n    for i in range(len(arr) + 1):\n        res.append(arr[i])\n    return res",
        "correct": "def add_elements(arr):\n    res = []\n    for i in range(len(arr)):\n        res.append(arr[i])\n    return res"
    },
    {
        "buggy": "def concatenate_strings(s1, s2):\n    return s1 + 10",
        "correct": "def concatenate_strings(s1, s2):\n    return s1 + str(10)"
    },
    {
        "buggy": "def compute_total(numbers):\n    for num in numbers:\n        total += num\n    return total",
        "correct": "def compute_total(numbers):\n    total = 0\n    for num in numbers:\n        total += num\n    return total"
    },
    {
        "buggy": "def parse_data(items):\n    for item in items\n        print(item)",
        "correct": "def parse_data(items):\n    for item in items:\n        print(item)"
    },
    {
        "buggy": "def check_palindrome(s):\n    res = s.lower()\n    return res",
        "correct": "def check_palindrome(s):\n    res = s.lower()\n    return (res, res == res[::-1])"
    },
    {
        "buggy": "def compute_fib(n):\n    return compute_fib(n-1) + compute_fib(n-2)",
        "correct": "def compute_fib(n):\n    if n <= 1:\n        return n\n    return compute_fib(n-1) + compute_fib(n-2)"
    },
    {
        "buggy": "def mod_exp(x, p):\n    return (x ** 2) % p + 1",
        "correct": "def mod_exp(x, p):\n    return (x ** 2) % p"
    },
    {
        "buggy": "def get_value(d, key):\n    return d[key]",
        "correct": "def get_value(d, key):\n    return d.get(key, None)"
    }
]

# Reasoning traces written in Qwen3's own thinking-mode style ("Okay, let's see..." + self-checks,
# blank line after </think>, matching the Qwen3 chat template). Used to extract the v_gist pull vector.
QWEN_REASONING_TRACES = [
    "<think>\nOkay, let's see. I need to write a function that processes the input sequence.\nFirst, I should handle the edge cases: an empty list or a single element.\nThen I'll initialize the accumulator variables before the loop so nothing is undefined.\nWait, the loop has to stay inside range(len(arr)), not len(arr) + 1, or I'll get an IndexError.\nLet me double-check the docstring examples... yes, the return type should match exactly.\n</think>\n\n```python\ndef solution():\n    pass\n```",
    "<think>\nAlright, the task is to parse nested parentheses.\nHmm, I can track the current depth with a counter and keep a separate max_depth variable.\nFor each character, increment on '(' and decrement on ')', updating max_depth as I go.\nI need to iterate over the characters directly so there is no index overflow.\nLet me verify with the example '(()())': depth goes 1, 2, 1, 2, 1, 0, so the max is 2. That's correct.\n</think>\n\n```python\ndef solution():\n    pass\n```"
]

class MaximumMarginVectorExtractor:
    def __init__(self, model: Optional[Any] = None, tokenizer: Optional[Any] = None, model_name: str = "Qwen/Qwen3-4B", device: Optional[str] = None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        fix_seeds(42)
        if model is not None and tokenizer is not None:
            self.model = model
            self.tokenizer = tokenizer
        else:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
            is_turing = "T4" in gpu_name or "2080" in gpu_name or "TU" in gpu_name
            dtype_to_use = torch.float16 if (is_turing or not torch.cuda.is_bf16_supported()) else torch.bfloat16

            load_kwargs = {
                "torch_dtype": dtype_to_use,
                "device_map": "auto" if self.device == "cuda" else None,
                "trust_remote_code": True
            }
            try:
                self.model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation="sdpa", **load_kwargs)
            except Exception:
                self.model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
            self.model.eval()

    def get_layer_activations(self, prompts: List[str], target_layer: int = 18) -> torch.Tensor:
        activations = []
        def hook_fn(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            activations.append(hidden[:, -1, :].detach().cpu().to(torch.float32))

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(hook_fn)
        try:
            with torch.inference_mode():
                for p in prompts:
                    formatted = format_chatml_prompt(self.tokenizer, p)
                    inputs = self.tokenizer(formatted, return_tensors="pt").to(self.model.device)
                    self.model(**inputs)
        finally:
            handle.remove()
        return torch.cat(activations, dim=0)

    def fit_maximum_margin_vector(self, buggy_prompts: List[str], correct_prompts: List[str], target_layer: int = 18) -> torch.Tensor:
        act_buggy = self.get_layer_activations(buggy_prompts, target_layer=target_layer)
        act_correct = self.get_layer_activations(correct_prompts, target_layer=target_layer)
        X = torch.cat([act_buggy, act_correct], dim=0).numpy()
        y = np.array([1] * len(act_buggy) + [-1] * len(act_correct))
        clf = SVC(kernel="linear", C=1.0)
        clf.fit(X, y)
        w = torch.tensor(clf.coef_[0], dtype=torch.float32)
        w_unit = w / torch.norm(w, p=2)
        return w_unit.to(self.model.device, dtype=self.model.dtype)

class QwenGistVectorExtractor:
    def __init__(self, model: Any, tokenizer: Any, device: Optional[str] = None):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")

    def extract_gist_vector(self, target_layer: int = 18, v_fail: Optional[torch.Tensor] = None) -> torch.Tensor:
        activations = []
        def hook_fn(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            activations.append(hidden[:, -1, :].detach().cpu().to(torch.float32))

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(hook_fn)
        try:
            with torch.inference_mode():
                for trace in QWEN_REASONING_TRACES:
                    inputs = self.tokenizer(trace, return_tensors="pt").to(self.model.device)
                    self.model(**inputs)
        finally:
            handle.remove()

        v_gist_raw = torch.cat(activations, dim=0).mean(dim=0)
        if v_fail is not None:
            v_fail_f32 = v_fail.detach().cpu().to(torch.float32)
            proj = (torch.dot(v_gist_raw, v_fail_f32) / torch.dot(v_fail_f32, v_fail_f32)) * v_fail_f32
            v_gist_raw = v_gist_raw - proj
        v_gist_unit = v_gist_raw / torch.norm(v_gist_raw, p=2)
        return v_gist_unit.to(self.model.device, dtype=self.model.dtype)

class SteeredGenerator:
    def __init__(self, model: Any, tokenizer: Any):
        self.model = model
        self.tokenizer = tokenizer

    def generate_steered(self, prompt: str, v_fail: torch.Tensor, target_layer: int = 18, alpha: float = 0.08, max_new_tokens: int = 1024) -> Tuple[str, Dict[str, Any]]:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        prompt_len = inputs.input_ids.shape[1]
        v_steer = -alpha * (v_fail / torch.norm(v_fail, p=2))
        s_fail_history = []

        def steering_hook(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            h_last = hidden[:, -1, :].clone()
            norm_h = torch.norm(h_last, p=2, dim=-1, keepdim=True) + 1e-8
            s_val = torch.sum((h_last / norm_h) * v_fail, dim=-1).item()
            s_fail_history.append(float(s_val))
            hidden[:, -1, :] = hidden[:, -1, :] + v_steer
            return (hidden,) + output[1:] if isinstance(output, tuple) else hidden

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(steering_hook)
        try:
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
        finally:
            handle.remove()

        gen_tokens = out[0][prompt_len:]
        text = self.tokenizer.decode(gen_tokens, skip_special_tokens=True)
        stats = {
            "total_tokens": len(gen_tokens),
            "s_fail_history": s_fail_history
        }
        return text, stats

class AdaptiveGatedSteeredGenerator:
    def __init__(self, model: Any, tokenizer: Any):
        self.model = model
        self.tokenizer = tokenizer

    def generate_gated(self, prompt: str, v_fail: torch.Tensor, target_layer: int = 18, alpha: float = 0.08, tau: float = 0.04, max_new_tokens: int = 1024) -> Tuple[str, Dict[str, Any]]:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        prompt_len = inputs.input_ids.shape[1]
        v_steer = -alpha * (v_fail / torch.norm(v_fail, p=2))
        gated_history = []
        s_fail_history = []

        def gating_hook(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            h_last = hidden[:, -1, :].clone()
            norm_h = torch.norm(h_last, p=2, dim=-1, keepdim=True) + 1e-8
            s_val = torch.sum((h_last / norm_h) * v_fail, dim=-1).item()
            s_fail_history.append(float(s_val))
            if s_val > tau:
                hidden[:, -1, :] = hidden[:, -1, :] + v_steer
                gated_history.append(1)
            else:
                gated_history.append(0)
            return (hidden,) + output[1:] if isinstance(output, tuple) else hidden

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(gating_hook)
        try:
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
        finally:
            handle.remove()

        gen_tokens = out[0][prompt_len:]
        text = self.tokenizer.decode(gen_tokens, skip_special_tokens=True)
        gating_ratio = (sum(gated_history) / max(len(gated_history), 1)) * 100.0
        stats = {
            "total_tokens": len(gen_tokens),
            "gating_ratio": gating_ratio,
            "s_fail_history": s_fail_history
        }
        return text, stats

class DualPushPullSteeredGenerator:
    def __init__(self, model: Any, tokenizer: Any):
        self.model = model
        self.tokenizer = tokenizer

    def generate_dual_steered(self, prompt: str, v_fail: torch.Tensor, v_gist: torch.Tensor, target_layer: int = 18, alpha: float = 0.08, beta: float = 0.015, cot_warmup_tokens: int = 200, max_new_tokens: int = 1024) -> Tuple[str, Dict[str, Any]]:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        prompt_len = inputs.input_ids.shape[1]
        v_push = -alpha * (v_fail / torch.norm(v_fail, p=2))
        v_pull = beta * (v_gist / torch.norm(v_gist, p=2))
        step_counter = [0]
        s_fail_history = []
        s_gist_history = []

        def dual_steering_hook(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            t_curr = step_counter[0]
            h_last = hidden[:, -1, :].clone()
            norm_h = torch.norm(h_last, p=2, dim=-1, keepdim=True) + 1e-8
            s_fail = torch.sum((h_last / norm_h) * v_fail, dim=-1).item()
            s_gist = torch.sum((h_last / norm_h) * v_gist, dim=-1).item()
            s_fail_history.append(float(s_fail))
            s_gist_history.append(float(s_gist))

            # Apply constant failure repulsion push
            delta = v_push
            # Apply reasoning CoT attraction pull during early generation
            if t_curr <= cot_warmup_tokens:
                delta = delta + v_pull
            hidden[:, -1, :] = hidden[:, -1, :] + delta
            step_counter[0] += 1
            return (hidden,) + output[1:] if isinstance(output, tuple) else hidden

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(dual_steering_hook)
        try:
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
        finally:
            handle.remove()

        gen_tokens = out[0][prompt_len:]
        text = self.tokenizer.decode(gen_tokens, skip_special_tokens=True)
        stats = {
            "total_tokens": len(gen_tokens),
            "s_fail_history": s_fail_history,
            "s_gist_history": s_gist_history
        }
        return text, stats

class EntropyEvaluator:
    def __init__(self, model: Any, tokenizer: Any):
        self.model = model
        self.tokenizer = tokenizer

    def compute_token_diagnostics(self, prompt: str, generated_text: str) -> Dict[str, Any]:
        full_text = prompt + generated_text
        inputs = self.tokenizer(full_text, return_tensors="pt").to(self.model.device)
        prompt_len = self.tokenizer(prompt, return_tensors="pt").input_ids.shape[1]
        with torch.inference_mode():
            outputs = self.model(**inputs)
            logits = outputs.logits[:, prompt_len - 1:-1, :].to(torch.float32)

        if logits.shape[1] == 0:
            return {"mean_entropy": 0.0, "r_rep": 0.0, "entropy_trajectory": []}

        probs = torch.softmax(logits, dim=-1)
        log_probs = torch.log2(torch.clamp(probs, min=1e-12))

        # Tensor-level negation before converting to list:
        entropy_tensor = -torch.sum(probs * log_probs, dim=-1)
        entropies = entropy_tensor.squeeze(0).cpu().tolist()
        mean_entropy = float(np.mean(entropies)) if len(entropies) > 0 else 0.0

        gen_ids = inputs.input_ids[0, prompt_len:].cpu().tolist()
        if len(gen_ids) < 3:
            r_rep = 0.0
        else:
            ngrams = [tuple(gen_ids[i:i+3]) for i in range(len(gen_ids) - 2)]
            unique_ngrams = len(set(ngrams))
            r_rep = 1.0 - (unique_ngrams / max(len(ngrams), 1))

        return {
            "mean_entropy": mean_entropy,
            "r_rep": float(r_rep),
            "entropy_trajectory": entropies
        }

## 🤖 Section 3: Load Qwen/Qwen3-4B Model & Extract Contrastive Vectors ($v_{\text{fail}}$ & $v_{\text{gist}}$)

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"
TARGET_LAYER = 18

print(f"Loading {MODEL_NAME} for Single-Layer (Layer {TARGET_LAYER}) 5-Way Steering Benchmark...")
extractor_vfail = MaximumMarginVectorExtractor(model_name=MODEL_NAME)
model = extractor_vfail.model
tokenizer = extractor_vfail.tokenizer

NUM_LAYERS = model.config.num_hidden_layers
D_MODEL = model.config.hidden_size
assert TARGET_LAYER < NUM_LAYERS, f"TARGET_LAYER={TARGET_LAYER} out of range for {NUM_LAYERS}-layer model"
print(f"Architecture: {NUM_LAYERS} decoder layers, d_model={D_MODEL} | Thinking mode: {ENABLE_THINKING}")

# Fit failure vector v_fail from 8 contrastive pairs at Layer 18
buggy_prompts = [p["buggy"] for p in CONTRASTIVE_PAIRS]
correct_prompts = [p["correct"] for p in CONTRASTIVE_PAIRS]
v_fail = extractor_vfail.fit_maximum_margin_vector(buggy_prompts, correct_prompts, target_layer=TARGET_LAYER)

# Extract Qwen gist vector v_gist (orthogonalized perp v_fail)
extractor_vgist = QwenGistVectorExtractor(model=model, tokenizer=tokenizer)
v_gist = extractor_vgist.extract_gist_vector(target_layer=TARGET_LAYER, v_fail=v_fail)

# Initialize Steering Generators & Evaluators
generator_static = SteeredGenerator(model=model, tokenizer=tokenizer)
generator_gated = AdaptiveGatedSteeredGenerator(model=model, tokenizer=tokenizer)
generator_dual = DualPushPullSteeredGenerator(model=model, tokenizer=tokenizer)
entropy_evaluator = EntropyEvaluator(model=model, tokenizer=tokenizer)

print(f"✅ Vectors successfully initialized at Layer {TARGET_LAYER}:")
print(f"  • v_fail norm: {torch.norm(v_fail, p=2):.4f}")
print(f"  • v_gist norm: {torch.norm(v_gist, p=2):.4f}")
print(f"  • Orthogonality dot product <v_fail, v_gist>: {torch.dot(v_fail, v_gist):.6f}")

## 📦 Section 4: Curated 100-Task Failure-Intensive HumanEval Dataset Loader (Zero Overlap)

In [ ]:
import gzip
import json
import urllib.request
from datasets import load_dataset

# 1. First 50 failure-focused HumanEval Tasks (Completed)
FIRST_50_HUMANEVAL_IDS = [
    "HumanEval/0", "HumanEval/1", "HumanEval/2", "HumanEval/3", "HumanEval/4",
    "HumanEval/5", "HumanEval/6", "HumanEval/7", "HumanEval/8", "HumanEval/9",
    "HumanEval/10", "HumanEval/12", "HumanEval/13", "HumanEval/14", "HumanEval/16",
    "HumanEval/20", "HumanEval/21", "HumanEval/23", "HumanEval/25", "HumanEval/27",
    "HumanEval/29", "HumanEval/31", "HumanEval/32", "HumanEval/36", "HumanEval/38",
    "HumanEval/49", "HumanEval/50", "HumanEval/55", "HumanEval/65", "HumanEval/67",
    "HumanEval/70", "HumanEval/78", "HumanEval/84", "HumanEval/89", "HumanEval/93",
    "HumanEval/94", "HumanEval/96", "HumanEval/108", "HumanEval/110", "HumanEval/112",
    "HumanEval/115", "HumanEval/118", "HumanEval/122", "HumanEval/123", "HumanEval/127",
    "HumanEval/130", "HumanEval/131", "HumanEval/132", "HumanEval/140", "HumanEval/141"
]

# 2. Curated 50 hardest, failure-prone, edge-case intensive uncompleted tasks (Tasks 51 to 100)
SECOND_50_FAILURE_INTENSIVE_IDS = [
    "HumanEval/19",  # sort_numbers: text-to-number mapping
    "HumanEval/37",  # sort_even: even indices sorting preserving odd
    "HumanEval/64",  # vowels_count: conditional 'y' string boundary rule
    "HumanEval/74",  # total_match: list char count comparison
    "HumanEval/81",  # numerical_letter_grade: float boundary conditions
    "HumanEval/83",  # starts_one_ends: combinatorial counting
    "HumanEval/85",  # add: odd indices with even values
    "HumanEval/88",  # sort_array: parity-dependent sort direction
    "HumanEval/91",  # is_bored: multi-delimiter sentence start parsing
    "HumanEval/95",  # check_dict_case: mixed string key case validation
    "HumanEval/97",  # multiply: unit digit negative product
    "HumanEval/98",  # count_upper: uppercase vowels at even indices
    "HumanEval/99",  # closest_integer: round away from zero boundary
    "HumanEval/102", # choose_num: range max even number
    "HumanEval/104", # unique_digits: no even digits filter
    "HumanEval/109", # move_one_ball: circular sorted shift
    "HumanEval/111", # histogram: frequency dictionary with multi-mode
    "HumanEval/114", # minSubArraySum: min Kadane subarray sum
    "HumanEval/116", # sort_array: binary popcount sort
    "HumanEval/119", # match_parens: open/close paren concatenation
    "HumanEval/120", # maximum: top k elements sorted
    "HumanEval/121", # solution: sum of odd elements in even positions
    "HumanEval/124", # valid_date: multi-condition date regex & day limits
    "HumanEval/125", # split_words: multi-delimiter fallback split
    "HumanEval/126", # is_sorted: sorted + max 2 duplicates
    "HumanEval/128", # prod_signs: magnitude sum times sign product
    "HumanEval/129", # minPath: grid path minimization (DFS/BFS)
    "HumanEval/133", # sum_squares: ceil squared sum
    "HumanEval/134", # check_if_last_char_is_a_letter: word boundary check
    "HumanEval/135", # can_arrange: largest inverted index
    "HumanEval/136", # largest_smallest_integers: signed extremes
    "HumanEval/137", # compare_one: mixed type float/comma string comparisons
    "HumanEval/138", # is_equal_to_sum_even: 4 positive even sum
    "HumanEval/139", # special_factorial: product of factorials
    "HumanEval/142", # sum_squares: index 3/4 divisibility
    "HumanEval/143", # words_in_sentence: prime length word filtering
    "HumanEval/144", # simplify: fraction multiplication whole number check
    "HumanEval/145", # order_by_points: signed digit sum sorting
    "HumanEval/146", # specialFilter: >10 with both first and last digits odd
    "HumanEval/147", # get_max_triples: modular arithmetic (a+b+c)%3==0
    "HumanEval/148", # bf: planet order traversal
    "HumanEval/149", # sorted_list_sum: even length string sorting
    "HumanEval/150", # x_or_y: prime check branching
    "HumanEval/151", # double_the_difference: odd integer square sum
    "HumanEval/152", # compare: prediction error array
    "HumanEval/153", # Strongest_Extension: case strength difference
    "HumanEval/154", # cycpattern_check: circular substring matching
    "HumanEval/156", # int_to_mini_roman: integer to Roman numeral conversion
    "HumanEval/158", # find_max: max unique chars string with tiebreak
    "HumanEval/160"  # do_algebra: operand and operator evaluation
]

def categorize_task(task_id: str) -> str:
    num = int(task_id.split("/")[-1]) if "/" in task_id else 0
    if num in [0, 4, 13, 21, 25, 31, 32, 49, 55, 67, 83, 96, 97, 122, 123, 130, 131, 138, 139, 147, 150, 151, 160]:
        return "Math & Number Theory"
    elif num in [1, 6, 7, 10, 12, 14, 16, 19, 23, 27, 29, 38, 50, 64, 65, 89, 91, 93, 112, 118, 124, 125, 134, 137, 140, 141, 143, 148, 149, 153, 154, 156, 158]:
        return "String Parsing & Manipulation"
    elif num in [2, 3, 5, 8, 9, 20, 36, 37, 70, 74, 78, 84, 85, 88, 94, 95, 98, 99, 102, 104, 108, 109, 110, 111, 112, 114, 115, 116, 119, 120, 121, 126, 127, 128, 132, 133, 135, 136, 142, 145, 146, 152]:
        return "Array & List Operations"
    elif num in [129]:
        return "Dynamic Programming & Recursion"
    else:
        return "Logic & Validation Rules"

print("Loading Official HumanEval Dataset...")
raw_humaneval_dict = {}
try:
    ds = load_dataset("openai_humaneval", split="test")
    for item in ds:
        raw_humaneval_dict[item["task_id"]] = {
            "task_id": item["task_id"],
            "benchmark": "HumanEval",
            "prompt": item["prompt"],
            "entry_point": item["entry_point"],
            "test": item["test"],
            "category": categorize_task(item["task_id"])
        }
except Exception as e:
    print(f"Loading via fallback URL: {e}")
    url = "https://github.com/openai/human-eval/raw/master/data/HumanEval.jsonl.gz"
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp:
        with gzip.GzipFile(fileobj=resp) as gz:
            for line in gz:
                item = json.loads(line.decode('utf-8'))
                raw_humaneval_dict[item["task_id"]] = {
                    "task_id": item["task_id"],
                    "benchmark": "HumanEval",
                    "prompt": item["prompt"],
                    "entry_point": item["entry_point"],
                    "test": item["test"],
                    "category": categorize_task(item["task_id"])
                }

# Assemble exact 100 HumanEval tasks (50 Completed + 50 Hard Uncompleted)
selected_tasks = []
# 1. Add first 50
for tid in FIRST_50_HUMANEVAL_IDS:
    if tid in raw_humaneval_dict:
        selected_tasks.append(raw_humaneval_dict[tid])

# 2. Add second 50 (Zero Overlap guaranteed)
for tid in SECOND_50_FAILURE_INTENSIVE_IDS:
    if tid in raw_humaneval_dict:
        selected_tasks.append(raw_humaneval_dict[tid])

print(f"✅ Successfully assembled exactly {len(selected_tasks)} failure-intensive HumanEval benchmark tasks:")
print(f"  • Tasks 1 to 50: Historical Baseline Failure Tasks (Original failure set)")
print(f"  • Tasks 51 to 100: Curated Failure-Intensive Tasks (Zero Overlap)")

## 🧪 Section 5: Universal HumanEval Code Extractor & Isolated Test Execution Engine

In [ ]:
def extract_clean_code(generation: str, entry_point: str, docstring_prompt: str = "") -> str:
    code = generation.strip()
    if "</think>" in code:
        code = code.split("</think>")[-1].strip()

    match = re.search(r'```python\s*(.*?)\s*(```|$)', code, re.DOTALL)
    if match:
        code = match.group(1).strip()
    else:
        match_any = re.search(r'```\s*(.*?)\s*(```|$)', code, re.DOTALL)
        if match_any:
            code = match_any.group(1).strip()

    lines = code.splitlines()
    clean_lines = []
    has_def = False

    for line in lines:
        if line.strip().startswith(f"def {entry_point}("):
            has_def = True
            clean_lines.append(line)
            continue

        if has_def:
            clean_lines.append(line)
        elif line.strip().startswith("import ") or line.strip().startswith("from "):
            clean_lines.append(line)

    clean_code = "\n".join(clean_lines).strip()
    if not clean_code or not has_def:
        if "def " in code:
            def_idx = code.find(f"def {entry_point}(")
            if def_idx != -1:
                clean_code = code[def_idx:].strip()
            else:
                first_def = code.find("def ")
                clean_code = code[first_def:].strip()
        else:
            clean_code = code

    return clean_code

def execute_unit_test(code: str, test_script: str, entry_point: str, docstring_prompt: str = "") -> Tuple[bool, str, str]:
    has_def = bool(re.search(r'def\s+' + re.escape(entry_point) + r'', code))
    if not has_def and docstring_prompt:
        full_code = f"import math\nimport re\nimport sys\nfrom typing import List, Tuple, Dict, Any, Optional, Union\n\n{docstring_prompt.strip()}\n    {code.replace('\n', '\n    ')}\n\n{test_script}"
    else:
        full_code = f"import math\nimport re\nimport sys\nfrom typing import List, Tuple, Dict, Any, Optional, Union\n\n{code}\n\n{test_script}"

    globals_dict = {
        "List": List, "Tuple": Tuple, "Dict": Dict, "Any": Any,
        "Optional": Optional, "Union": Union, "math": math, "re": re
    }
    output_buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(output_buffer):
            exec(full_code, globals_dict)
        return True, "SUCCESS", "Passed clean execution."
    except SyntaxError as err:
        return False, "SyntaxError", f"SyntaxError at line {err.lineno}: {err.msg}"
    except NameError as err:
        return False, "NameError", f"NameError: {str(err)}"
    except TypeError as err:
        return False, "TypeError", f"TypeError: {str(err)}"
    except IndexError as err:
        return False, "IndexError", f"IndexError: {str(err)}"
    except AssertionError as err:
        return False, "AssertionError", "AssertionError (Logic Failure): Output did not match assertion."
    except Exception as err:
        return False, type(err).__name__, f"{type(err).__name__}: {str(err)}"


## 🧪 Section 6: Live 5-Way Steering Benchmark Execution Loop (All 100 Tasks Live, Resumable)

In [ ]:
ALPHA = 0.08
BETA = 0.015
TAU = round(2.0 / math.sqrt(D_MODEL), 3)  # Theoretical 2-sigma threshold for a random unit-vector cosine (d_model=2560 -> 0.04)
TOKEN_BUDGET = 1024  # Expanded token budget to prevent truncation

# The VibeThinker notebook pre-loaded Tasks 1-63 results; those belong to a different model,
# so every task is evaluated live for Qwen3.
variants = ["vanilla", "neg_prompt", "static_steered", "gated_steered", "dual_qwen"]
CHECKPOINT_FILE = "humaneval_100_steering_checkpoint_qwen3.json"

variant_results = {v: {"pass": [], "time": [], "tokens": [], "entropy": [], "r_rep": [], "gating": [], "categories": {}} for v in variants}
benchmark_logs = []
start_idx = 0

# Resume a previous partial run (T4 Colab sessions can disconnect mid-benchmark)
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f_chk:
        chk = json.load(f_chk)
    if chk.get("model_name") == MODEL_NAME and chk.get("enable_thinking") == ENABLE_THINKING:
        variant_results = chk["variant_results"]
        benchmark_logs = chk["benchmark_logs"]
        start_idx = chk["completed_tasks"]
        print(f"♻️ Resumed from checkpoint: {start_idx} tasks already evaluated.")

print(f"🚀 Starting live evaluation loop on {len(selected_tasks[start_idx:])} HumanEval tasks (Tasks {start_idx + 1} to {len(selected_tasks)}), TAU={TAU}...")

for idx, task in enumerate(selected_tasks[start_idx:], start=start_idx + 1):
    task_id = task["task_id"]
    entry_point = task["entry_point"]
    raw_prompt = task["prompt"]
    test_script = task["test"]
    category = task["category"]
    benchmark = task.get("benchmark", "HumanEval")

    prompt_chatml = format_chatml_prompt(tokenizer, raw_prompt)
    prompt_neg = format_negative_system_prompt(tokenizer, raw_prompt)

    task_entry = {"task_id": task_id, "benchmark": benchmark, "entry_point": entry_point, "category": category, "outputs": {}}

    # 1. Vanilla Baseline
    t0 = time.time()
    inputs_v = tokenizer(prompt_chatml, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out_v = model.generate(**inputs_v, max_new_tokens=TOKEN_BUDGET, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    t_v = time.time() - t0
    txt_v = tokenizer.decode(out_v[0][inputs_v.input_ids.shape[1]:], skip_special_tokens=True)
    code_v = extract_clean_code(txt_v, entry_point, raw_prompt)
    pass_v, err_v, trace_v = execute_unit_test(code_v, test_script, entry_point, raw_prompt)
    diag_v = entropy_evaluator.compute_token_diagnostics(prompt_chatml, txt_v)
    tok_v = out_v[0].shape[0] - inputs_v.input_ids.shape[1]

    variant_results["vanilla"]["pass"].append(1 if pass_v else 0)
    variant_results["vanilla"]["time"].append(t_v)
    variant_results["vanilla"]["tokens"].append(tok_v)
    variant_results["vanilla"]["entropy"].append(diag_v["mean_entropy"])
    variant_results["vanilla"]["r_rep"].append(diag_v["r_rep"])
    variant_results["vanilla"]["gating"].append(0.0)
    variant_results["vanilla"]["categories"][err_v] = variant_results["vanilla"]["categories"].get(err_v, 0) + 1
    task_entry["outputs"]["vanilla"] = {
        "pass": pass_v, "err": err_v, "trace": trace_v, "txt": txt_v, "code": code_v,
        "time": t_v, "entropy": diag_v["mean_entropy"], "r_rep": diag_v["r_rep"],
        "entropy_traj": diag_v["entropy_trajectory"]
    }

    # 2. Negative System Prompting
    t0 = time.time()
    inputs_n = tokenizer(prompt_neg, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out_n = model.generate(**inputs_n, max_new_tokens=TOKEN_BUDGET, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    t_n = time.time() - t0
    txt_n = tokenizer.decode(out_n[0][inputs_n.input_ids.shape[1]:], skip_special_tokens=True)
    code_n = extract_clean_code(txt_n, entry_point, raw_prompt)
    pass_n, err_n, trace_n = execute_unit_test(code_n, test_script, entry_point, raw_prompt)
    diag_n = entropy_evaluator.compute_token_diagnostics(prompt_neg, txt_n)
    tok_n = out_n[0].shape[0] - inputs_n.input_ids.shape[1]

    variant_results["neg_prompt"]["pass"].append(1 if pass_n else 0)
    variant_results["neg_prompt"]["time"].append(t_n)
    variant_results["neg_prompt"]["tokens"].append(tok_n)
    variant_results["neg_prompt"]["entropy"].append(diag_n["mean_entropy"])
    variant_results["neg_prompt"]["r_rep"].append(diag_n["r_rep"])
    variant_results["neg_prompt"]["gating"].append(0.0)
    variant_results["neg_prompt"]["categories"][err_n] = variant_results["neg_prompt"]["categories"].get(err_n, 0) + 1
    task_entry["outputs"]["neg_prompt"] = {
        "pass": pass_n, "err": err_n, "trace": trace_n, "txt": txt_n, "code": code_n,
        "time": t_n, "entropy": diag_n["mean_entropy"], "r_rep": diag_n["r_rep"],
        "entropy_traj": diag_n["entropy_trajectory"]
    }

    # 3. Static Failure Steering (Layer 18, alpha=0.08)
    t0 = time.time()
    txt_s, stat_s = generator_static.generate_steered(prompt_chatml, v_fail=v_fail, target_layer=TARGET_LAYER, alpha=ALPHA, max_new_tokens=TOKEN_BUDGET)
    t_s = time.time() - t0
    code_s = extract_clean_code(txt_s, entry_point, raw_prompt)
    pass_s, err_s, trace_s = execute_unit_test(code_s, test_script, entry_point, raw_prompt)
    diag_s = entropy_evaluator.compute_token_diagnostics(prompt_chatml, txt_s)
    tok_s = stat_s.get("total_tokens", len(tokenizer.encode(txt_s)))

    variant_results["static_steered"]["pass"].append(1 if pass_s else 0)
    variant_results["static_steered"]["time"].append(t_s)
    variant_results["static_steered"]["tokens"].append(tok_s)
    variant_results["static_steered"]["entropy"].append(diag_s["mean_entropy"])
    variant_results["static_steered"]["r_rep"].append(diag_s["r_rep"])
    variant_results["static_steered"]["gating"].append(100.0)
    variant_results["static_steered"]["categories"][err_s] = variant_results["static_steered"]["categories"].get(err_s, 0) + 1
    task_entry["outputs"]["static_steered"] = {
        "pass": pass_s, "err": err_s, "trace": trace_s, "txt": txt_s, "code": code_s,
        "time": t_s, "entropy": diag_s["mean_entropy"], "r_rep": diag_s["r_rep"],
        "s_fail_traj": stat_s.get("s_fail_history", []),
        "entropy_traj": diag_s["entropy_trajectory"]
    }

    # 4. Adaptive Gated Failure Steering (Layer 18, alpha=0.08, tau=0.04)
    t0 = time.time()
    txt_g, stat_g = generator_gated.generate_gated(prompt_chatml, v_fail=v_fail, target_layer=TARGET_LAYER, alpha=ALPHA, tau=TAU, max_new_tokens=TOKEN_BUDGET)
    t_g = time.time() - t0
    code_g = extract_clean_code(txt_g, entry_point, raw_prompt)
    pass_g, err_g, trace_g = execute_unit_test(code_g, test_script, entry_point, raw_prompt)
    diag_g = entropy_evaluator.compute_token_diagnostics(prompt_chatml, txt_g)
    tok_g = stat_g.get("total_tokens", len(tokenizer.encode(txt_g)))
    gate_g = stat_g.get("gating_ratio", 0.0)

    variant_results["gated_steered"]["pass"].append(1 if pass_g else 0)
    variant_results["gated_steered"]["time"].append(t_g)
    variant_results["gated_steered"]["tokens"].append(tok_g)
    variant_results["gated_steered"]["entropy"].append(diag_g["mean_entropy"])
    variant_results["gated_steered"]["r_rep"].append(diag_g["r_rep"])
    variant_results["gated_steered"]["gating"].append(gate_g)
    variant_results["gated_steered"]["categories"][err_g] = variant_results["gated_steered"]["categories"].get(err_g, 0) + 1
    task_entry["outputs"]["gated_steered"] = {
        "pass": pass_g, "err": err_g, "trace": trace_g, "txt": txt_g, "code": code_g,
        "time": t_g, "gating_ratio": gate_g, "entropy": diag_g["mean_entropy"], "r_rep": diag_g["r_rep"],
        "s_fail_traj": stat_g.get("s_fail_history", []),
        "entropy_traj": diag_g["entropy_trajectory"]
    }

    # 5. Dual Push-Pull Qwen Steering ⭐ (Layer 18, alpha=0.08, beta=0.015)
    t0 = time.time()
    txt_d, stat_d = generator_dual.generate_dual_steered(prompt_chatml, v_fail=v_fail, v_gist=v_gist, target_layer=TARGET_LAYER, alpha=ALPHA, beta=BETA, max_new_tokens=TOKEN_BUDGET)
    t_d = time.time() - t0
    code_d = extract_clean_code(txt_d, entry_point, raw_prompt)
    pass_d, err_d, trace_d = execute_unit_test(code_d, test_script, entry_point, raw_prompt)
    diag_d = entropy_evaluator.compute_token_diagnostics(prompt_chatml, txt_d)
    tok_d = stat_d.get("total_tokens", len(tokenizer.encode(txt_d)))

    variant_results["dual_qwen"]["pass"].append(1 if pass_d else 0)
    variant_results["dual_qwen"]["time"].append(t_d)
    variant_results["dual_qwen"]["tokens"].append(tok_d)
    variant_results["dual_qwen"]["entropy"].append(diag_d["mean_entropy"])
    variant_results["dual_qwen"]["r_rep"].append(diag_d["r_rep"])
    variant_results["dual_qwen"]["gating"].append(100.0)
    variant_results["dual_qwen"]["categories"][err_d] = variant_results["dual_qwen"]["categories"].get(err_d, 0) + 1
    task_entry["outputs"]["dual_qwen"] = {
        "pass": pass_d, "err": err_d, "trace": trace_d, "txt": txt_d, "code": code_d,
        "time": t_d, "entropy": diag_d["mean_entropy"], "r_rep": diag_d["r_rep"],
        "s_fail_traj": stat_d.get("s_fail_history", []),
        "s_gist_traj": stat_d.get("s_gist_history", []),
        "entropy_traj": diag_d["entropy_trajectory"]
    }

    benchmark_logs.append(task_entry)

    sym_v = "✅" if pass_v else "❌"
    sym_n = "✅" if pass_n else "❌"
    sym_s = "✅" if pass_s else "❌"
    sym_g = "✅" if pass_g else "❌"
    sym_d = "✅" if pass_d else "❌"

    bench_tag = f"[{benchmark}]"
    print(
        f"[{idx:03d}/{len(selected_tasks):03d}] {bench_tag:<12} {task_id:<22} ({entry_point:<20}) | "
        f"Vanilla: {sym_v} | NegPrompt: {sym_n} | Static: {sym_s} | RepGated: {sym_g} | DualQwen: {sym_d} ⭐",
        flush=True
    )

    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f_chk:
        json.dump({
            "model_name": MODEL_NAME,
            "enable_thinking": ENABLE_THINKING,
            "completed_tasks": idx,
            "variant_results": variant_results,
            "benchmark_logs": benchmark_logs
        }, f_chk)
    if idx % 10 == 0 or idx == len(selected_tasks):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

print("\n" + "=" * 115)
print("      ALL 100-TASK 5-WAY STEERING EVALUATIONS COMPLETE!")
print("=" * 115)

## 📈 Section 7: Comprehensive Performance Table & 5-Panel Statistical Dashboard

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Print Comprehensive Performance & Statistical Table (WITHOUT mean S_fail scalar)
print("\n" + "=" * 135)
print(f"{'Variant Name':<35} | {'Pass@1 Accuracy':<18} | {'95% Wilson CI':<20} | {'Shannon H(X)':<14} | {'Repetition R_rep':<18} | {'Throughput':<15}")
print("=" * 135)

vanilla_passes = variant_results["vanilla"]["pass"]
n_total = len(vanilla_passes)

summary_table_data = []

def compute_wilson_ci(k: int, n: int, confidence: float = 0.95) -> Tuple[float, float]:
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    spread = z * math.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
    lower = max(0.0, (centre - spread) / denominator) * 100
    upper = min(1.0, (centre + spread) / denominator) * 100
    return lower, upper

def compute_mcnemar_exact(b_baseline_only: int, c_steered_only: int) -> Tuple[float, float]:
    total_discordant = b_baseline_only + c_steered_only
    if total_discordant == 0:
        return 0.0, 1.0
    chi2 = ((abs(b_baseline_only - c_steered_only) - 0.5) ** 2) / total_discordant
    p_val = stats.chi2.sf(chi2, df=1)
    return chi2, p_val

labels_map = {
    "vanilla": "1. Vanilla Baseline",
    "neg_prompt": "2. Negative System Prompting",
    "static_steered": "3. Static Failure Steered (a=0.08)",
    "gated_steered": "4. Adaptive RepGated (tau=0.04)",
    "dual_qwen": "5. Dual Push-Pull Qwen ⭐"
}

for v in variants:
    passes = variant_results[v]["pass"]
    times = variant_results[v]["time"]
    tokens = variant_results[v]["tokens"]
    entropies = variant_results[v]["entropy"]
    r_reps = variant_results[v]["r_rep"]

    k = sum(passes)
    acc = (k / n_total) * 100 if n_total > 0 else 0.0
    ci_low, ci_high = compute_wilson_ci(k, n_total)

    mean_h = float(np.mean(entropies)) if entropies else 0.0
    mean_rep = float(np.mean(r_reps)) if r_reps else 0.0

    total_time = sum(times)
    total_tok = sum(tokens)
    throughput = (total_tok / total_time) if total_time > 0 else 0.0

    summary_table_data.append({
        "key": v,
        "name": labels_map[v],
        "k": k, "n": n_total,
        "acc": acc,
        "ci": (ci_low, ci_high),
        "entropy": mean_h,
        "r_rep": mean_rep,
        "throughput": throughput
    })

    print(
        f"{labels_map[v]:<35} | "
        f"{k:02d}/{n_total:02d} ({acc:6.2f}%)   | "
        f"[{ci_low:5.1f}%, {ci_high:5.1f}%]     | "
        f"{mean_h:6.3f} bits   | "
        f"{mean_rep:6.4f}           | "
        f"{throughput:6.2f} tok/s"
    )

print("=" * 135)

# McNemar Paired Exact Significance Tests vs Vanilla
print("\n" + "=" * 80)
print("             MCNEMAR PAIRED EXACT SIGNIFICANCE TESTS (vs. Vanilla)")
print("=" * 80)
for item in summary_table_data[1:]:
    v_key = item["key"]
    steered_passes = variant_results[v_key]["pass"]
    b_only = sum(1 for b, s in zip(vanilla_passes, steered_passes) if b == 1 and s == 0)
    s_only = sum(1 for b, s in zip(vanilla_passes, steered_passes) if b == 0 and s == 1)
    chi2, p_val = compute_mcnemar_exact(b_only, s_only)
    sig_str = "Statistically Significant (p < 0.05) ✅" if p_val < 0.05 else "Marginal / Trend (p >= 0.05)"
    print(f"• {item['name']:<35}: Vanilla-Only={b_only:02d}, Steered-Only={s_only:02d} | p-value={p_val:.4f} ({sig_str})")
print("=" * 80)

# Render 5-Panel Statistical Dashboard
fig, axs = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle(f"Qwen3-4B: 5-Way Steering Benchmark Suite on 100 HumanEval Tasks", fontsize=16, fontweight="bold", y=0.98)

colors = ["#4A5568", "#ED8936", "#3182CE", "#805AD5", "#38A169"]
variant_display_names = ["Vanilla", "NegPrompt", "Static", "RepGated", "DualQwen ⭐"]

# Panel 1: Pass@1 Accuracy with 95% Wilson CIs
accs = [d["acc"] for d in summary_table_data]
err_low = [d["acc"] - d["ci"][0] for d in summary_table_data]
err_high = [d["ci"][1] - d["acc"] for d in summary_table_data]
yerrs = [err_low, err_high]

bars1 = axs[0, 0].bar(variant_display_names, accs, color=colors, yerr=yerrs, capsize=5, edgecolor="black", alpha=0.9)
axs[0, 0].set_title("1. Pass@1 Accuracy (95% Wilson CIs)", fontweight="bold")
axs[0, 0].set_ylabel("Pass@1 Accuracy (%)")
axs[0, 0].set_ylim(0, 100)
for bar in bars1:
    yval = bar.get_height()
    axs[0, 0].text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=10)

# Panel 2: Shannon Information Entropy H(X)
entropies = [d["entropy"] for d in summary_table_data]
bars2 = axs[0, 1].bar(variant_display_names, entropies, color=colors, edgecolor="black", alpha=0.9)
axs[0, 1].set_title("2. Shannon Entropy H(X) (Cognitive Uncertainty)", fontweight="bold")
axs[0, 1].set_ylabel("Entropy (bits)")
for bar in bars2:
    yval = bar.get_height()
    axs[0, 1].text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=10)

# Panel 3: 3-Gram Repetition Ratio
reps = [d["r_rep"] for d in summary_table_data]
bars3 = axs[0, 2].bar(variant_display_names, reps, color=colors, edgecolor="black", alpha=0.9)
axs[0, 2].set_title("3. 3-Gram Repetition Ratio R_rep", fontweight="bold")
axs[0, 2].set_ylabel("Repetition Ratio")
for bar in bars3:
    yval = bar.get_height()
    axs[0, 2].text(bar.get_x() + bar.get_width()/2, yval + 0.001, f"{yval:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=10)

# Panel 4: AST Failure Taxonomy
category_names = ["SUCCESS", "AssertionError", "SyntaxError", "NameError", "TypeError", "IndexError"]
cat_colors = ["#38A169", "#E53E3E", "#D69E2E", "#3182CE", "#805AD5", "#DD6B20"]
bottoms = np.zeros(len(variants))
for c_idx, cat in enumerate(category_names):
    vals = [variant_results[v]["categories"].get(cat, 0) for v in variants]
    axs[1, 0].bar(variant_display_names, vals, bottom=bottoms, label=cat, color=cat_colors[c_idx], edgecolor="white")
    bottoms += np.array(vals)
axs[1, 0].set_title("4. AST Execution Error Taxonomy", fontweight="bold")
axs[1, 0].set_ylabel("Total Task Count")
axs[1, 0].legend(loc="upper right", fontsize=8)

# Panel 5: Generation Throughput (tok/s)
throughputs = [d["throughput"] for d in summary_table_data]
bars5 = axs[1, 1].bar(variant_display_names, throughputs, color=colors, edgecolor="black", alpha=0.9)
axs[1, 1].set_title("5. Generation Throughput (tokens/sec) on Tesla T4", fontweight="bold")
axs[1, 1].set_ylabel("Tokens / Second")
for bar in bars5:
    yval = bar.get_height()
    axs[1, 1].text(bar.get_x() + bar.get_width()/2, yval + 0.5, f"{yval:.1f}", ha="center", va="bottom", fontweight="bold", fontsize=10)

# Panel 6: Steering Intervention Dynamics
axs[1, 2].axis("off")
info_text = (
    "STEERING INTERVENTION MECHANICS:\n\n"
    "• Single-Layer Residual Hook at Layer 18\n"
    "• Double L2 Unit-Sphere Normalization\n"
    "• Failure Repulsion: a = 0.08\n"
    "• Qwen Attraction: b = 0.015 (t <= 200)\n"
    f"• Theoretical 2s Gating: tau = {TAU}\n\n"
    "ACCELERATION ENGINE:\n"
    "• Native Turing FP16 Tensor Cores\n"
    "• PyTorch SDPA Memory-Efficient Attention\n"
    "• Zero-overhead O(1) Residual Hook"
)
axs[1, 2].text(0.1, 0.5, info_text, fontsize=11, family="monospace", va="center", bbox=dict(boxstyle="round,pad=1", facecolor="#EDF2F7", edgecolor="#CBD5E0"))

plt.tight_layout()
plt.show()

## 🔍 Section 8: Interactive Task-by-Task Trajectory Comparator & Quality Inspector

In [ ]:
def plot_task_trajectory_comparison(task_id: str):
    found = [entry for entry in benchmark_logs if entry["task_id"] == task_id]
    if not found:
        print(f"Task {task_id} not found in evaluated logs.")
        return

    t_log = found[0]
    print("=" * 100)
    print(f"           STEP-BY-STEP LATENT TRAJECTORY COMPARISON: {task_id} ({t_log['entry_point']})")
    print(f"           Category: {t_log['category']}")
    print("=" * 100)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    colors_dict = {
        "vanilla": "#4A5568",
        "neg_prompt": "#ED8936",
        "static_steered": "#3182CE",
        "gated_steered": "#805AD5",
        "dual_qwen": "#38A169"
    }

    for v_name in variants:
        v_out = t_log["outputs"].get(v_name, {})
        status_sym = "✅ PASS" if v_out.get("pass", False) else "❌ FAIL"
        ent_traj = v_out.get("entropy_traj", [])
        if ent_traj:
            ax1.plot(ent_traj[:150], label=f"{labels_map[v_name]} ({status_sym})", color=colors_dict[v_name], linewidth=1.8)

        s_fail_traj = v_out.get("s_fail_traj", [])
        if s_fail_traj:
            ax2.plot(s_fail_traj[:150], label=f"{labels_map[v_name]}", color=colors_dict[v_name], linewidth=1.8)

    ax1.set_title(f"Per-Token Shannon Entropy Trajectory: {task_id}", fontweight="bold")
    ax1.set_xlabel("Token Decoding Step (t)")
    ax1.set_ylabel("Entropy H(p_t) (bits)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8)

    ax2.set_title(f"Per-Token Latent Failure Cosine Projection: {task_id}", fontweight="bold")
    ax2.set_xlabel("Token Decoding Step (t)")
    ax2.set_ylabel("Cosine Margin S_{t, fail}")
    ax2.axhline(y=TAU, color="red", linestyle="--", alpha=0.7, label=f"Gating Threshold tau={TAU}")
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

# Example: Inspect trajectory on recovered failure task HumanEval/67
plot_task_trajectory_comparison("HumanEval/67")